In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import *
import json

pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.width", None)        # + better readability

In [82]:
cols_to_keep = [
    "id", 
    "formattedAddress", 
    "rating", 
    "googleMapsUri",
    "websiteUri",
    "businessStatus",
    "priceLevel",
    "displayName",
    "takeout",
    "delivery",
    "dineIn",
    "servesBreakfast",
    "servesLunch",
    "servesDinner",
    "servesBrunch",
    "servesDessert",
    "primaryType",
    "priceRange",
    "servesVegetarianFood",
]
re_index = [
    "id", 
    "name",
    "formattedAddress", 
    "primaryType",
    "googleMapsUri",
    "websiteUri",
    "businessStatus",
    "rating", 
    "takeout",
    "delivery",
    "dineIn",
    "servesBreakfast",
    "servesLunch",
    "servesDinner",
    "servesBrunch",
    "servesDessert",
    "servesVegetarianFood",
    "priceLevel",
    "startPrice",
    "endPrice"
]

In [83]:
df_raw = pd.read_json(RAW_DATA_DIR / 'copenhagen-10pct-sample.raw.json')
df_raw = df_raw[cols_to_keep]

# unpack the name
df_raw["name"] = df_raw["displayName"].apply(lambda d: d["text"] if isinstance(d, dict) else d)
df_raw = df_raw.drop(['displayName'], axis=1)

# unpack price_range
price_range = pd.json_normalize(df_raw['priceRange']).drop(columns=['startPrice.currencyCode', 'endPrice.currencyCode'])
price_range.rename({'startPrice.units': 'startPrice', 'endPrice.units': 'endPrice'}, axis='columns', inplace=True)
df_raw = pd.concat([df_raw.drop(columns='priceRange'), price_range], axis=1)

# remove restaurants out of order
df_raw = df_raw[df_raw["businessStatus"] == "OPERATIONAL"]

# re-index
df_cleaned = df_raw.reindex(columns=re_index)
df_cleaned.head()

,id,name,formattedAddress,primaryType,googleMapsUri,websiteUri,businessStatus,rating,takeout,delivery,dineIn,servesBreakfast,servesLunch,servesDinner,servesBrunch,servesDessert,servesVegetarianFood,priceLevel,startPrice,endPrice
0,ChIJAV1_vX1SUkYRifHQGs0-33o,Pizze Di Napo Hellerup,"Strandvejen 149, 2900 Hellerup, Denmark",pizza_restaurant,https://maps.google.com/?cid=88538644430728032...,https://platia.dk/pizze-di-napo-hellerup,OPERATIONAL,4.5,1.0,NaN,1.0,0.0,1.0,1.0,0.0,1.0,NaN,PRICE_LEVEL_MODERATE,1,100
1,ChIJ7RIaXXpQUkYRxGD3xBRRltk,McDonald's,"Tempovej 1, 2750 Ballerup, Denmark",fast_food_restaurant,https://maps.google.com/?cid=15678808302382702...,https://www.mcdonalds.com/dk/da-dk/location/ba...,OPERATIONAL,3.5,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,PRICE_LEVEL_MODERATE,100,200
2,ChIJq6p-9w9TUkYRGJ7PByHSDYY,Byens Pizza,"Nørregade 45c, 1165 København, Denmark",restaurant,https://maps.google.com/?cid=96596078150863826...,http://byens-pizza.dk/?page_id=2065,OPERATIONAL,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ChIJz3lHpeRSUkYRRNH0_-MSMhU,Restaurant Taormina v/Mukuntharaj Karunaratnam,"Slagelsegade 24, 2100 København, Denmark",restaurant,https://maps.google.com/?cid=15273039940931300...,https://taormina.dk/,OPERATIONAL,4.3,NaN,0.0,1.0,0.0,NaN,1.0,0.0,1.0,NaN,PRICE_LEVEL_MODERATE,200,300
4,ChIJBVGBtQFTUkYR7R1FSqjW1kg,Barnetsfest,"Blegdamsvej 24, 2200 København, Denmark",family_restaurant,https://maps.google.com/?cid=52486184340357851...,http://www.barnetsfest.dk/,OPERATIONAL,NaN,NaN,0.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN


In [84]:
csv_path = RUNS_DATA_DIR / 'responses'
files = sorted(csv_path.glob('20260917-*'))

dfs = []
for f in files:
    df_llm = pd.read_csv(f, usecols=['model', 'answer'])
    df_llm['answer'] = df_llm['answer'].apply(json.loads)
    df_answer = pd.json_normalize(df_llm['answer'])
    df_llm = df_llm.drop(columns=['answer']).join(df_answer)
    df_llm['model'] = df_llm['model'].str.split('/').str[1]
    dfs.append(df_llm)

df_llm = pd.concat(dfs, ignore_index=True)
df_llm.head()

,model,id,name,hasFoundMenu,menuLink,format,source
0,claude-sonnet-5,ChIJAV1_vX1SUkYRifHQGs0-33o,Pizze Di Napo Hellerup,True,https://pizzedinapohellerup.dk/,HTML,OFFICIAL_WEBSITE
1,claude-sonnet-5,ChIJ7RIaXXpQUkYRxGD3xBRRltk,McDonald's,True,https://www.mcdonalds.com/dk/da-dk/vores-menu....,HTML,OFFICIAL_WEBSITE
2,claude-sonnet-5,ChIJq6p-9w9TUkYRGJ7PByHSDYY,Byens Pizza,True,http://byens-pizza.dk/?page_id=2065,HTML,OFFICIAL_WEBSITE
3,claude-sonnet-5,ChIJz3lHpeRSUkYRRNH0_-MSMhU,Restaurant Taormina v/Mukuntharaj Karunaratnam,True,https://taormina.dk/menu/,HTML,OFFICIAL_WEBSITE
4,claude-sonnet-5,ChIJBVGBtQFTUkYR7R1FSqjW1kg,Barnetsfest,False,,,


In [85]:
model_menulinks = df_llm.pivot_table(index='id', columns='model', values='menuLink', aggfunc='first')
model_menulinks

model,claude-sonnet-5,gemini-3.7-flash,gpt-5.6-luna,gpt-5.6-terra
id,,,,
ChIJ-7iMbc1TUkYR066Hz1SQ-EI,https://www.restaurantalf.dk/,https://restaurantalf.dk/,https://www.restaurantalf.dk/,https://www.restaurantalf.dk/
ChIJ-QrW55VTUkYRiO5rYIpA5PE,https://durumbar-frb.dk/,https://durumbar78.dk/,https://durumbar2000.dk/,https://www.ubereats.com/dk-en/store/durum-bar...
ChIJ-XF3Wu1TUkYRl25fQ2IL8Ps,https://royalbagel.dk/royal-bagel-valby-langga...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://www.ubereats.com/dk-en/store/royal-bag...
ChIJ-YewRThSUkYRU-ZX1KiCAr0,https://nordvestpizzaria.dk/,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...
ChIJ-ao52_ZNUkYRU11OTne3ec8,https://www.joejuice.com/,https://www.joejuice.com/menu,https://www.joejuice.com/?market=dk,https://www.ubereats.com/dk-en/store/joe-%26-t...
...,...,...,...,...
ChIJzSGiCwVTUkYRM2XmxLIIRqc,https://www.hallernes.dk/en/standard-smorrebrod/,https://hallernes.dk/,https://www.hallernes.dk/faste-smorrebrod/,https://torvehallerne.hallernes.dk/take-away
ChIJzVHSkstTUkYRp1DtTvwjdlI,,,,
ChIJzX87fApZUkYR3MgmhXk4aXI,,https://www.eatonline.dk/restaurant/190/big-m-...,https://www.eatonline.dk/restaurant/190/big-m-...,https://www.eatonline.dk/restaurant/190/big-m-...


In [86]:
merged = model_menulinks.merge(df_cleaned, on='id')
merged.info()
merged.head()

<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    414 non-null    str    
 1   claude-sonnet-5       414 non-null    str    
 2   gemini-3.7-flash      414 non-null    str    
 3   gpt-5.6-luna          414 non-null    str    
 4   gpt-5.6-terra         414 non-null    str    
 5   name                  414 non-null    str    
 6   formattedAddress      414 non-null    str    
 7   primaryType           414 non-null    str    
 8   googleMapsUri         414 non-null    str    
 9   websiteUri            337 non-null    str    
 10  businessStatus        414 non-null    str    
 11  rating                386 non-null    float64
 12  takeout               313 non-null    float64
 13  delivery              281 non-null    float64
 14  dineIn                378 non-null    float64
 15  servesBreakfast       170 non-null

,id,claude-sonnet-5,gemini-3.7-flash,gpt-5.6-luna,gpt-5.6-terra,name,formattedAddress,primaryType,googleMapsUri,websiteUri,businessStatus,rating,takeout,delivery,dineIn,servesBreakfast,servesLunch,servesDinner,servesBrunch,servesDessert,servesVegetarianFood,priceLevel,startPrice,endPrice
0,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,https://www.restaurantalf.dk/,https://restaurantalf.dk/,https://www.restaurantalf.dk/,https://www.restaurantalf.dk/,Restaurant Alf,"Gl. Kongevej 86A, 1850 Frederiksberg, Denmark",restaurant,https://maps.google.com/?cid=48257656946609148...,https://restaurantalf.dk/,OPERATIONAL,4.5,NaN,0.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN
1,ChIJ-QrW55VTUkYRiO5rYIpA5PE,https://durumbar-frb.dk/,https://durumbar78.dk/,https://durumbar2000.dk/,https://www.ubereats.com/dk-en/store/durum-bar...,Durum Bar 2000,"Peter Bangs Vej 78, 2000 Frederiksberg, Denmark",restaurant,https://maps.google.com/?cid=17430127420898012...,https://durumbar78.dk/,OPERATIONAL,4.6,1.0,NaN,1.0,NaN,1.0,1.0,NaN,NaN,NaN,NaN,1,100
2,ChIJ-XF3Wu1TUkYRl25fQ2IL8Ps,https://royalbagel.dk/royal-bagel-valby-langga...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://www.ubereats.com/dk-en/store/royal-bag...,Royal Bagel,"Valby Langgade 75, st tv, 2500, 2500 København...",bagel_shop,https://maps.google.com/?cid=18154022614595497...,https://royalbagel.dk/?utm_source=GMBwebsite&u...,OPERATIONAL,4.1,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PRICE_LEVEL_INEXPENSIVE,100,200
3,ChIJ-YewRThSUkYRU-ZX1KiCAr0,https://nordvestpizzaria.dk/,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...,Nordvest Pizzaria,"Frederiksborgvej 96, 2400 København, Denmark",pizza_restaurant,https://maps.google.com/?cid=13619591884750448...,https://www.eatonline.dk/restaurant/172/nord-v...,OPERATIONAL,3.6,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,PRICE_LEVEL_INEXPENSIVE,1,100
4,ChIJ-ao52_ZNUkYRU11OTne3ec8,https://www.joejuice.com/,https://www.joejuice.com/menu,https://www.joejuice.com/?market=dk,https://www.ubereats.com/dk-en/store/joe-%26-t...,Joe & The Juice,"Gentoftegade 64, 2820 Gentofte, Denmark",juice_shop,https://maps.google.com/?cid=14950182161075821...,http://www.joejuice.com/,OPERATIONAL,3.9,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,PRICE_LEVEL_MODERATE,100,200


In [87]:
discarded_restaurants = [
    "bakery",
    "banquet_hall",
    "bar",
    "bar_and_grill",
    "cafe",
    "car_wash",
    "catering_service",
    "cocktail_bar",
    "coffee_shop",
    "dessert_shop",
    "food_court",
    "gas_station",
    "hot_dog_restaurant",
    "hot_dog_stand",
    "hotel",
    "irish_pub",
    "juice_shop",
    "kebab_shop",
    "meal_takeaway",
    "movie_theater",
    "museum",
    "pastry_shop",
    "point_of_interest",
    "pub",
    "salad_shop",
    "sandwich_shop",
    "shawarma_restaurant",
    "wine_bar",
]

restaurants_to_keep = [
    "argentinian_restaurant",
    "asian_fusion_restaurant",
    "asian_restaurant",
    "bagel_shop",
    "bistro",
    "brazilian_restaurant",
    "brunch_restaurant",
    "buffet_restaurant",
    "chicken_restaurant",
    "chinese_restaurant",
    "danish_restaurant",
    "diner",
    "ethiopian_restaurant",
    "family_restaurant",
    "fast_food_restaurant",
    "fine_dining_restaurant",
    "french_restaurant",
    "fusion_restaurant",
    "greek_restaurant",
    "halal_restaurant",
    "hamburger_restaurant",
    "hawaiian_restaurant",
    "indian_restaurant",
    "indonesian_restaurant",
    "italian_restaurant",
    "japanese_restaurant",
    "korean_restaurant",
    "mexican_restaurant",
    "pakistani_restaurant",
    "persian_restaurant",
    "pizza_delivery",
    "pizza_restaurant",
    "restaurant",
    "seafood_restaurant",
    "spanish_restaurant",
    "sushi_restaurant",
    "thai_restaurant",
    "turkish_restaurant",
    "vegetarian_restaurant",
    "vietnamese_restaurant",
]

In [88]:
filtered_types = merged[merged['primaryType'].isin(restaurants_to_keep)].reset_index(drop=True)
filtered_types.head()

,id,claude-sonnet-5,gemini-3.7-flash,gpt-5.6-luna,gpt-5.6-terra,name,formattedAddress,primaryType,googleMapsUri,websiteUri,businessStatus,rating,takeout,delivery,dineIn,servesBreakfast,servesLunch,servesDinner,servesBrunch,servesDessert,servesVegetarianFood,priceLevel,startPrice,endPrice
0,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,https://www.restaurantalf.dk/,https://restaurantalf.dk/,https://www.restaurantalf.dk/,https://www.restaurantalf.dk/,Restaurant Alf,"Gl. Kongevej 86A, 1850 Frederiksberg, Denmark",restaurant,https://maps.google.com/?cid=48257656946609148...,https://restaurantalf.dk/,OPERATIONAL,4.5,NaN,0.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN
1,ChIJ-QrW55VTUkYRiO5rYIpA5PE,https://durumbar-frb.dk/,https://durumbar78.dk/,https://durumbar2000.dk/,https://www.ubereats.com/dk-en/store/durum-bar...,Durum Bar 2000,"Peter Bangs Vej 78, 2000 Frederiksberg, Denmark",restaurant,https://maps.google.com/?cid=17430127420898012...,https://durumbar78.dk/,OPERATIONAL,4.6,1.0,NaN,1.0,NaN,1.0,1.0,NaN,NaN,NaN,NaN,1,100
2,ChIJ-XF3Wu1TUkYRl25fQ2IL8Ps,https://royalbagel.dk/royal-bagel-valby-langga...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://www.ubereats.com/dk-en/store/royal-bag...,Royal Bagel,"Valby Langgade 75, st tv, 2500, 2500 København...",bagel_shop,https://maps.google.com/?cid=18154022614595497...,https://royalbagel.dk/?utm_source=GMBwebsite&u...,OPERATIONAL,4.1,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PRICE_LEVEL_INEXPENSIVE,100,200
3,ChIJ-YewRThSUkYRU-ZX1KiCAr0,https://nordvestpizzaria.dk/,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...,https://www.eatonline.dk/restaurant/172/nord-v...,Nordvest Pizzaria,"Frederiksborgvej 96, 2400 København, Denmark",pizza_restaurant,https://maps.google.com/?cid=13619591884750448...,https://www.eatonline.dk/restaurant/172/nord-v...,OPERATIONAL,3.6,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,PRICE_LEVEL_INEXPENSIVE,1,100
4,ChIJ-fk2NRBTUkYRyhj40xrfESA,https://www.aterre.dk/menuer,http://www.aterre.dk/,https://www.earlybird.dk/restaurant/a-terre,https://starwinelist.com/storage/food_menus/21...,à terre,"Tordenskjoldsgade 11, 1055 København, Denmark",french_restaurant,https://maps.google.com/?cid=23108733901361706...,http://www.aterre.dk/,OPERATIONAL,4.8,0.0,0.0,1.0,NaN,1.0,1.0,NaN,1.0,NaN,NaN,1000,NaN


In [91]:
manual_inspection = filtered_types[['id', 'name', 'websiteUri', 'claude-sonnet-5', 'gemini-3.7-flash', 'gpt-5.6-luna','gpt-5.6-terra', 'googleMapsUri']]
manual_inspection.to_csv(RUNS_DATA_DIR / 'filtered_types_manual_inspection.csv')